# Apply babyseg to the OpenNeuro open dataset

In [1]:
# The path to the data and where you want to save it. 
# We will keep it in the BIDS format and add the segmentations to the derivatives file
import os
import subprocess
from pathlib import Path
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np


mri_path =  "sub-01/anat/sub-01_T1w.nii.gz" # path into the dataset

derivatives_outpath = "../open_data/OpenNeuro/derivatives/sub-01/anat" # where to save the segmentation

os.makedirs(derivatives_outpath,exist_ok=True) # make the segmentation path

babyseg_path = "derivatives/sub-01/anat/babyseg_out.nii.gz"



In [8]:
import os
import subprocess
import sys
from pathlib import Path

babyseg_home = (Path("..") / "babyseg").resolve()
wrapper = babyseg_home / "wrapper.py"
openneuro_root = (Path("..") / "open_data" / "OpenNeuro").resolve()

if not wrapper.exists():
    raise FileNotFoundError(f"wrapper.py not found at: {wrapper}")

env = os.environ.copy()
env["BABYSEG_MNT"] = str(openneuro_root)
# Default to CPU container for broader compatibility. Override by setting BABYSEG_TAG in env.
env["BABYSEG_TAG"] = env.get("BABYSEG_TAG", "0.0")
env["BABYSEG_SIF"] = str(babyseg_home)

# Use a single thread to reduce memory pressure on Docker Desktop defaults.
cmd = [sys.executable, str(wrapper), "-j", "1", "-o", babyseg_path, mri_path]
print("Running:", " ".join(cmd))
print("Using BABYSEG_TAG:", env["BABYSEG_TAG"])

result = subprocess.run(
    cmd,
    env=env,
    text=True,
    capture_output=True,
    check=False,
 )

if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    if result.stderr:
        print(result.stderr)
    if result.returncode == 137:
        raise RuntimeError(
            "babyseg process was killed (exit 137). Increase Docker memory, keep -j 1, or close other Docker workloads."
        )
    raise RuntimeError(f"babyseg run failed with exit code {result.returncode}")

Running: c:\Users\spn486\Documents\GitHub\OpenInfantMRI_workshop\.venv\Scripts\python.exe C:\Users\spn486\Documents\GitHub\OpenInfantMRI_workshop\babyseg\wrapper.py -j 1 -o derivatives/sub-01/anat/babyseg_out.nii.gz sub-01/anat/sub-01_T1w.nii.gz
Using BABYSEG_TAG: 0.0
Applying environment variable BABYSEG_MNT="C:\Users\spn486\Documents\GitHub\OpenInfantMRI_workshop\open_data\OpenNeuro"
Applying environment variable BABYSEG_TAG="0.0"
Applying environment variable BABYSEG_SIF="C:\Users\spn486\Documents\GitHub\OpenInfantMRI_workshop\babyseg"
Running BabySeg version "0.0" from https://hub.docker.com/u/freesurfer
Selected "C:\Program Files\Docker\Docker\resources\bin\docker.EXE" to manage containers
Will bind /mnt in container to BABYSEG_MNT="C:\Users\spn486\Documents\GitHub\OpenInfantMRI_workshop\open_data\OpenNeuro"
Command: C:\Program Files\Docker\Docker\resources\bin\docker.EXE run --rm -v C:\Users\spn486\Documents\GitHub\OpenInfantMRI_workshop\open_data\OpenNeuro:/mnt freesurfer/babyse

RuntimeError: babyseg process was killed (exit 137). Increase Docker memory, keep -j 1, or close other Docker workloads.

# Look at the segmentation

In [ ]:
# Load in image and segmentation
image = nib.load(str(mri_path)).get_fdata()
seg = nib.load(str(babyseg_path)).get_fdata()


# Generate the color map:
labels = np.unique(seg) 
label_map = {label: i for i, label in enumerate(labels)}
seg_mapped = np.vectorize(label_map.get)(seg)

# Find the midplane
x_mid,y_mid,z_mid = image.shape[0] // 2 -10, image.shape[1] // 2, image.shape[2] // 2

# plot the image
fig, axes = plt.subplots(ncols=3, figsize=(12, 12))


def overlay(ax, img_slice, seg_slice):

    # Show MRI scan
    ax.imshow(img_slice, cmap="gray")
    #  mask background so it doesn't wash out MRI
    seg_masked = np.ma.masked_where(seg_slice == 0, seg_slice)
    # plot masked segmentation
    ax.imshow(seg_masked,cmap="tab20",alpha=0.5)
    ax.axis("off")


overlay(axes[0], np.rot90(image[x_mid, :, :]), np.rot90(seg_mapped[x_mid, :, :]))
overlay(axes[1], np.rot90(image[:, y_mid, :]), np.rot90(seg_mapped[:, y_mid, :]))
overlay(axes[2], np.rot90(image[:, :, z_mid]), np.rot90(seg_mapped[:, :, z_mid]))
